
# Transform Race Data


 1. Read bronze `races` table
 1. Keep only the columns required for analytics (Drop `url` column)
 1. Standardise column names using snake_case (`raceName` → `race_name`, `circuitId` → `circuit_id`)
 1. Rename columns to make them more meaningful (`date` → `race_date`)
 1. Remove duplicate records
 1. Transform values of column `race_name` to Title Case
 1. Write the transformed data to silver `races` table


![formula1-silver-data-erd.png](./formula1-silver-data-erd.png "formula1-silver-data-erd.png")

In [0]:
%run ../00-common/1.Environment-Config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.races"
silver_table = f"{catalog_name}.{silver_schema}.races"


## Step 1 Read bronze races table

In [0]:
races_df = spark.read.table(bronze_table)

In [0]:
display(races_df)

#### Step 2 - Keep only the columns required for analytics (Drop url column)

In [0]:
races_selected_df = races_df.select(
    "season",
    "round",
    "raceName",
    "date",
    "circuitId",
    "ingestion_timestamp",
    "source_file"
)

In [0]:
from pyspark.sql import functions as F

In [0]:
races_selected_df= races_df.select(
    F.col("season"),
    F.col("round"),
    F.col("raceName"),
    F.col("date"),
    F.col("CircuitId"),
    F.col("ingestion_timestamp"),
    F.col("source_file")
)

#### Step 3 & 4 - Standardise Column Names
- Standardise column names using snake_case (`raceId` → `race_id`, `raceName` → `race_name`)
- Rename columns to make them more meaningful (`lat` → `latitude`, `long` → `longitude`)

In [0]:
races_renamed_df= (
    races_selected_df
            .withColumnRenamed("raceName", "race_name")
            .withColumnRenamed("circuitId", "circuit_id")
            .withColumnRenamed("date", "race_date")    
)

Possible option to rename
circuits_renamed_df= (
    circuits_selected_df
        .withColumnsRenamed({
            "circuitId": "circuit_id",
            "circuitName": "circuit_name",
            "lat": "latitude",
            "long": "longitude"
        })
)

In [0]:
display(races_renamed_df)

In [0]:
#null validation
from pyspark.sql.functions import col, sum

# Generate a count of nulls for each column
null_counts = races_renamed_df.select([sum(col(c).isNull().cast("int")).alias(c) for c in races_renamed_df.columns])
null_counts.show()


 ## Step 5 - Filter out rows where race_id is null (business key validation)

In [0]:
races_distinct_df = races_renamed_df.dropDuplicates(["season","round"])

In [0]:
display(races_distinct_df)

## Step 6 - Remove duplicate records

### Step 7 - Transform values of columns `race_name` Title Case

In [0]:
races_finals_df = (
    races_distinct_df
    .withColumn('race_name',F.initcap(F.col("race_name")))
)

In [0]:
display(races_finals_df)

## Step 8 - Write the transformed data to silver `races` table

In [0]:
(
    races_finals_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))